        # 🪙 S2　支線：一千枚硬幣的陷阱（多重檢定）
        **統計冒險之旅 2026**　｜　支線（選修，隨時可做）　｜　支線任務　｜　🏅 100 XP

        📖 ISLP Ch13


        ### 🎯 這一關你會學到
        - 模擬 1000 次檢定看假發現
- Bonferroni 與 Benjamini–Hochberg
- 什麼是 p-hacking

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/stats-quest-2026/rc/v1.1.0-rc.1/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。
        > 🎲 這門課的答案常常是小數：任務會告訴你要把答案存進哪個變數，檢查時允許小小的誤差；切分、抽樣、模型請照題目用 `random_state=42`。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  統計冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins, math, warnings
warnings.filterwarnings("ignore")

_LEVEL = "S2"
_COURSE_NAMESPACE = "stats-quest-2026-datama"
_PREFIX = "SQ"
_TASKS = ["S2-1", "S2-2", "S2-3", "S2-4"]
_XP_EACH = 25
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_sq_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

# ---------------- 判分器 2.0 ----------------
class _Miss(Exception):
    pass

def 抓變數(ns, name, 型別=None):
    """從任務格執行後的變數取值；沒有就給友善訊息。"""
    if name not in ns:
        raise _Miss(f"我找不到變數 {name}，請確認你有把答案存進名字叫 {name} 的變數（大小寫要一樣）。")
    v = ns[name]
    if 型別 is not None and not isinstance(v, 型別):
        raise _Miss(f"{name} 的型別看起來不對（目前是 {type(v).__name__}）。")
    return v

def _num(v):
    try:
        import numpy as _np
        if hasattr(v, "item"): v = v.item()
    except Exception:
        pass
    return float(v)

def 約等於(v, 目標, 容差=None, 相對=0.01):
    """數值容差：|v-目標| <= 容差（預設為 目標 的 1%，且至少 1e-9）"""
    try:
        x = _num(v)
    except Exception:
        return False
    if x != x:   # NaN
        return False
    tol = 容差 if 容差 is not None else max(abs(目標) * 相對, 1e-9)
    return abs(x - 目標) <= tol

def 資料框像(obj, 列=None, 欄=None, 含欄位=None, 種類="DataFrame"):
    """檢查 DataFrame / Series：列數、欄數、必須包含的欄位；回傳 (ok, 訊息)"""
    import pandas as _pd
    if 種類 == "DataFrame" and not isinstance(obj, _pd.DataFrame):
        return False, f"這應該是一個 DataFrame（目前是 {type(obj).__name__}）。"
    if 種類 == "Series" and not isinstance(obj, _pd.Series):
        return False, f"這應該是一個 Series（目前是 {type(obj).__name__}）。"
    if 列 is not None and len(obj) != 列:
        return False, f"列數應該是 {列}，目前是 {len(obj)}。"
    if 欄 is not None and getattr(obj, "shape", (0, 0))[1] != 欄:
        return False, f"欄數應該是 {欄}，目前是 {obj.shape[1]}。"
    if 含欄位:
        cols = list(obj.columns) if hasattr(obj, "columns") else list(obj.index)
        missing = [c for c in 含欄位 if c not in cols]
        if missing:
            return False, "缺少欄位：" + "、".join(map(str, missing))
    return True, ""


class _NeedMoreInput(Exception):
    pass

_BUILTIN_NAMES = ("sum", "list", "dict", "set", "str", "int", "float", "max", "min", "len",
                  "print", "type", "range", "sorted", "abs", "round", "tuple", "map", "filter",
                  "open", "format", "all", "any", "zip", "bool", "next", "chr", "ord", "id")

_HIST = builtins.__dict__.setdefault("_sq_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_sq_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_sq_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

_CALL = re.compile(r"\s*(檢查|通關密語|全部檢查)\s*\(")

def _clean_cell(cell):
    return "\n".join(ln for ln in cell.splitlines() if not _CALL.match(ln))

def _is_mine(cell):
    s = cell.strip()
    if not s:
        return False
    if "#@title" in s or "任務定義(" in s or "_sq_" in s:
        return False
    if _CALL.match(s):
        return False
    return True

def _find_cells(tid):
    marker = "# 🎯 任務 " + tid
    marked = free = None
    im = ifree = -1
    for i, cell in enumerate(_history()):
        if not _is_mine(cell):
            continue
        if marker in cell:
            marked, im = cell, i
        elif "🎯 任務" not in cell:
            free, ifree = cell, i
    return marked, im, free, ifree

def _describe(src):
    body = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
    if not body:
        return "（空白）"
    first = body[0].strip()
    return ("%s%s（共 %d 行）" % (first[:52], "…" if len(first) > 52 else "", len(body)))

def _fig_info(_plt):
    out = []
    try:
        for n in _plt.get_fignums():
            f = _plt.figure(n)
            for ax in f.get_axes():
                out.append(dict(title=ax.get_title() or "", xlabel=ax.get_xlabel() or "", ylabel=ax.get_ylabel() or "",
                                n_lines=len(ax.lines), n_patches=len(ax.patches), n_collections=len(ax.collections),
                                legend=bool(ax.get_legend())))
    except Exception:
        pass
    return out

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        run.shadowed = []
        for _n in _BUILTIN_NAMES:
            _b = getattr(builtins, _n, None)
            if _n in ns and _b is not None and ns[_n] is not _b:
                ns.pop(_n, None)
                run.shadowed.append(_n)
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        run.figs = []
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                run.figs = _fig_info(_plt)
                _plt.show = _orig_show
                _plt.close("all")
        return buf.getvalue(), ns
    run.src = src
    run.figs = []
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _fix_shadowed():
    try:
        ns = get_ipython().user_ns
    except Exception:
        ns = globals()
    bad = []
    for n in _BUILTIN_NAMES:
        b = builtins.__dict__.get(n)
        if b is not None and n in ns and ns[n] is not b:
            del ns[n]
            bad.append(n)
    return bad

def _progress():
    done = 0
    total = 0
    for t in _TASKS:
        total += 1
        if _PASSED.get(t):
            done += 1
    bar = "■" * done + "□" * (total - done)
    return f"[{bar}] {done}/{total}"

def _run_check(tid, src):
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        return False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。", []
    except _Miss as e:
        return False, str(e), getattr(run, "shadowed", [])
    except Exception:
        tb = traceback.format_exc().strip().splitlines()[-1]
        return False, "程式執行時發生錯誤 → " + tb, getattr(run, "shadowed", [])
    ok, extra = (result, "") if isinstance(result, bool) else result
    return ok, extra, getattr(run, "shadowed", [])

def _pass(tid):
    first = not _PASSED.get(tid)
    _PASSED[tid] = True
    print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")

def 檢查(tid):
    _shadow = _fix_shadowed()
    tid = builtins.str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    marked, im, free, ifree = _find_cells(tid)
    if marked is None and free is None:
        print(f"❌ 這次執行階段裡，我找不到你寫的程式。")
        print(f"   👉 請先按「# 🎯 任務 {tid}」那一格左邊的 ▶ 執行它，再執行這一格。")
        print("   （如果剛剛重新啟動過執行階段，上面每一格都要重跑一次，包含最上面的魔法工具箱）")
        return
    order = []
    if marked is not None:
        order.append(("標記", marked))
    if free is not None and ifree > im:
        order.append(("最後執行", free))
    if not order:
        order = [("最後執行", free)]
    tried = []
    for kind, src in order:
        ok, extra, shadowed = _run_check(tid, _clean_cell(src))
        tried.append((kind, src, extra, shadowed))
        if ok:
            _pass(tid)
            if extra:
                print("   💬 " + str(extra))
            if _shadow:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(_shadow)} 拿來當變數名了，我已經幫你還原。")
                print("      建議換個名字（例如 total、items），不然後面的程式會出現很難懂的錯誤。")
            if kind == "最後執行":
                print(f"   ℹ️ 你的程式最上面少了「# 🎯 任務 {tid}」那一行，我是用你最後執行的那一格判分的。")
                print("      把那一行加回去，之後的檢查會更準確。")
            if shadowed:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(shadowed)} 拿來當變數名了，判分時我先幫你還原。")
            if all(_PASSED.get(t) for t in _TASKS):
                print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
            return
    kind, src, extra, shadowed = tried[0]
    print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
    if extra:
        print("   💬 " + str(extra))
    if _HINTS.get(tid):
        print("   💡 提示：" + _HINTS[tid])
    _sh = _shadow + [n for n in shadowed if n not in _shadow]
    if _sh:
        print(f"   ⚠️ 你把內建名稱 {'、'.join(_sh)} 拿來當變數名了（我已還原），這會造成很難懂的錯誤，請改名後重跑那一格。")
    print("   🔎 我判分的是這一段程式：" + _describe(src))
    print(f"      如果這不是你剛剛寫的版本 → 確認第一行的「# 🎯 任務 {tid}」有保留，並重新執行那一格，再按檢查。")

def 全部檢查():
    """出錯或重新啟動執行階段後，重跑完所有任務格，再用這個一次驗收整關。"""
    _fix_shadowed()
    print(f"🔁 重新檢查 {_LEVEL} 的 {len(_TASKS)} 個任務…")
    todo = []
    for t in _TASKS:
        marked, im, free, ifree = _find_cells(t)
        if marked is None and free is None:
            todo.append(t)
            continue
        檢查(t)
    if todo:
        print("⏭️ 這次還沒執行過的任務：" + "、".join(todo))
        print("   先按那幾格左邊的 ▶ 執行，再回來執行 全部檢查()。")

def 通關密語():
    _fix_shadowed()
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_COURSE_NAMESPACE}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：{_PREFIX}-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

try:
    import numpy as _np_, pandas as _pd_
    _np_.random.seed(42)
except Exception:
    pass
print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_S2_1(run):
    out, ns = run()
    n = int(抓變數(ns, "顯著數"))
    if abs(n - 43) > 15: return (False, "顯著數 = (p值們 < 0.05).sum()，應該在 50 附近。")
    return (約等於(抓變數(ns, "假發現率"), n / 1000, 1e-6), "假發現率 = 顯著數 / 1000。")
任務定義("S2-1", _check_S2_1, 提示="(p值們 < 0.05).sum()。")

def _check_S2_2(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "門檻"), 0.00005, 1e-9): return (False, "門檻 = 0.05 / 1000。")
    if int(抓變數(ns, "Bonferroni顯著數")) > 3: return (False, "Bonferroni 後應該幾乎沒有顯著。")
    return (bool(抓變數(ns, "寧可錯殺")) == True, "寧可錯殺 = Bonferroni顯著數 < 5。")
任務定義("S2-2", _check_S2_2, 提示="0.05 / 1000。")

def _check_S2_3(run):
    out, ns = run()
    if abs(int(抓變數(ns, "不修正數")) - 139) > 15: return (False, "不修正數 = (p_mix < 0.05).sum()。")
    if abs(int(抓變數(ns, "BH數")) - 87) > 15: return (False, "BH數：method='fdr_bh'。")
    return (int(抓變數(ns, "BH假貨數")) <= 20 and int(抓變數(ns, "BH數")) > int(抓變數(ns, "Bonferroni數")), "BH 應該比 Bonferroni 抓到更多，且假貨只有一小部分。")
任務定義("S2-3", _check_S2_3, 提示="method='fdr_bh'。")

def _check_S2_4(run):
    out, ns = run()
    return (約等於(抓變數(ns, "至少一個顯著的比例"), 0.6415, 0.12), "應該接近 1 - 0.95^20 ≈ 0.64：試 20 種切法，六成以上會「找到」顯著。")
任務定義("S2-4", _check_S2_4, 提示="range(20)。")

In [ ]:
import numpy as np, pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests

## 🪙 一千枚硬幣的陷阱
Day 1 學過：p < 0.05 代表「如果其實沒差，光靠運氣看到這結果的機會不到 5%」。
那如果我**做 1,000 次檢定**呢？就算全部都沒差，光靠運氣也會有 5%——約 50 個——「顯著」。這就是**多重檢定（multiple testing）**的陷阱：
> 📬 股市分析師寄 1,024 封信，每封預測不同走勢，10 天後總有一個人收到「連續 10 次全中」的信——不是他神，是抽樣。

先模擬看看：1,000 次「兩組其實一樣」的 t 檢定。

In [ ]:
#@title 🈶 中文字型設定（畫圖前先執行；Colab 初次約 20～40 秒）
import glob, shutil, subprocess, sys, matplotlib
from matplotlib import font_manager

_font_globs = [
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc',
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.otf',
    'C:/Windows/Fonts/msjh*.ttc',
]
if sys.platform.startswith('linux') and shutil.which('apt-get'):
    if not any(glob.glob(pattern) for pattern in _font_globs[:2]):
        try:
            subprocess.run(
                ['apt-get', '-qq', 'install', '-y', 'fonts-noto-cjk'],
                check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
        except (FileNotFoundError, subprocess.CalledProcessError) as error:
            raise RuntimeError('無法自動安裝中文字型；請確認網路後重新執行本格。') from error
for pattern in _font_globs:
    for path in glob.glob(pattern):
        try:
            font_manager.fontManager.addfont(path)
        except (OSError, RuntimeError):
            pass

_available_fonts = {font.name for font in font_manager.fontManager.ttflist}
_preferred_fonts = [
    'Noto Sans TC', 'Noto Sans CJK TC',
    'Microsoft JhengHei', 'Microsoft JhengHei UI', 'PingFang TC',
    'Noto Sans CJK JP', 'Arial Unicode MS',
]
_chinese_font = next((name for name in _preferred_fonts if name in _available_fonts), None)
if _chinese_font is None:
    raise RuntimeError('找不到可顯示繁體中文的字型；請安裝 Noto Sans CJK 後重新執行本格。')
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = [_chinese_font, 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
print(f"✅ 中文字型設定完成：{_chinese_font}")

In [ ]:
rng = np.random.default_rng(42)
p值們 = []
for _ in range(1000):
    a = rng.normal(0, 1, 30); b = rng.normal(0, 1, 30)      # 兩組其實一模一樣
    p值們.append(stats.ttest_ind(a, b).pvalue)
p值們 = np.array(p值們)
print("其實沒差，卻顯著（p < 0.05）的檢定數：", (p值們 < 0.05).sum(), "/ 1000")
plt.hist(p值們, bins=20); plt.title("沒差的時候，p 值是均勻分布的"); plt.xlabel("p 值"); plt.show()

## 兩種解法
| 方法 | 想控制什麼 | 做法 | 個性 |
|---|---|---|---|
| **Bonferroni** | FWER：整組**一個都不能錯**的機率 | 門檻改成 0.05 / m | 寧可錯殺，功率很低 |
| **Benjamini–Hochberg（BH）** | FDR：宣告顯著的裡面**假貨的比例** | p 值排序後動態門檻 | 容忍少量假貨，換更多發現 |

`multipletests(p, alpha=0.05, method="bonferroni")`、`method="fdr_bh"`。

In [ ]:
# 真實情境：1,000 個檢定裡有 100 個是真的有差（效果 1.0），900 個沒差
rng = np.random.default_rng(42)
p_mix, 真相 = [], []
for i in range(1000):
    真 = i >= 900
    a = rng.normal(0, 1, 30); b = rng.normal(1.0 if 真 else 0, 1, 30)
    p_mix.append(stats.ttest_ind(a, b).pvalue); 真相.append(真)
p_mix, 真相 = np.array(p_mix), np.array(真相)
for name, mask in [("不修正 p<0.05", p_mix < 0.05), ("Bonferroni", multipletests(p_mix, 0.05, "bonferroni")[0]), ("BH (FDR)", multipletests(p_mix, 0.05, "fdr_bh")[0])]:
    print(f"{name:>14}：宣告顯著 {mask.sum():>3} 個，其中假貨 {(mask & ~真相).sum():>3} 個，抓到真的 {(mask & 真相).sum():>3} / 100")

### 🎯 任務 S2-1　沒差也會顯著

用種子 42 的 `rng` 模擬 1,000 次「兩組一樣」的 t 檢定（每組 30 筆、N(0,1)），存成陣列 `p值們`；算 `顯著數`（p < 0.05 的個數）與 `假發現率`（顯著數 / 1000）。

In [ ]:
# 🎯 任務 S2-1　沒差也會顯著（請保留這一行）
rng = np.random.default_rng(42)
p值們 = []
for _ in range(1000):
    a = rng.normal(0, 1, 30); b = rng.normal(0, 1, 30)
    p值們.append(stats.ttest_ind(a, b).pvalue)
p值們 = np.array(p值們)
顯著數 = ???
假發現率 = 顯著數 / 1000
print(顯著數, 假發現率)

In [ ]:
檢查("S2-1")   # ◀ 執行這一格，看看任務 S2-1 有沒有過關

### 🎯 任務 S2-2　Bonferroni

算出 Bonferroni 的 `門檻`（0.05 / 1000）與 `Bonferroni顯著數`（p 值 < 門檻的個數），並把 `寧可錯殺` 設成布林值（Bonferroni 顯著數是否小於 5）。

In [ ]:
# 🎯 任務 S2-2　Bonferroni（請保留這一行）
門檻 = ???
Bonferroni顯著數 = int((p值們 < 門檻).sum())
寧可錯殺 = Bonferroni顯著數 < 5
print(門檻, Bonferroni顯著數, 寧可錯殺)

In [ ]:
檢查("S2-2")   # ◀ 執行這一格，看看任務 S2-2 有沒有過關

### 🎯 任務 S2-3　BH：容忍少量假貨換更多發現

沿用範例的 `p_mix`、`真相`：算出三種做法宣告顯著的個數 `不修正數`、`Bonferroni數`、`BH數`，以及 BH 宣告的假貨數 `BH假貨數`。

In [ ]:
# 🎯 任務 S2-3　BH：容忍少量假貨換更多發現（請保留這一行）
不修正數 = int((p_mix < 0.05).sum())
Bonferroni數 = int(multipletests(p_mix, 0.05, "bonferroni")[0].sum())
BH顯著 = multipletests(p_mix, 0.05, ???)[0]
BH數 = int(BH顯著.sum())
BH假貨數 = int((BH顯著 & ~真相).sum())
print(不修正數, Bonferroni數, BH數, BH假貨數)

In [ ]:
檢查("S2-3")   # ◀ 執行這一格，看看任務 S2-3 有沒有過關

### 🎯 任務 S2-4　p-hacking：切 20 種看哪種顯著

模擬 200 次「研究者對同一份沒差的資料試 20 種切法、只報告最小的 p 值」：用種子 42 的 `rng`，每次做 20 個 t 檢定（各組 30 筆、N(0,1)）取最小 p 值，算 `至少一個顯著的比例`（最小 p < 0.05 的次數 / 200）。

In [ ]:
# 🎯 任務 S2-4　p-hacking：切 20 種看哪種顯著（請保留這一行）
rng = np.random.default_rng(42)
命中 = 0
for _ in range(200):
    最小p = min(stats.ttest_ind(rng.normal(0, 1, 30), rng.normal(0, 1, 30)).pvalue for _ in range(???))
    if 最小p < 0.05:
        命中 += 1
至少一個顯著的比例 = 命中 / 200
print(至少一個顯著的比例, "→ 理論值 1 - 0.95**20 =", round(1 - 0.95 ** 20, 3))

In [ ]:
檢查("S2-4")   # ◀ 執行這一格，看看任務 S2-4 有沒有過關

---
## 🔑 通關密語
　你已經知道「顯著」是可以被試出來的——所以要先寫好假設，再看資料。
全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**支線完成！** 回入口網頁的「補給站 → 支線任務」蓋章。

回到入口網頁：https://johnnychao.github.io/stats-quest-2026/rc/v1.1.0-rc.1/